# Lab 3 - Part 1: Text Visualization & Classical Representations

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import string

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
from PIL import Image

print("Setup complete!")

## Part A: Loading and Exploring the 20 Newsgroups Dataset

In [ ]:
splits = {'train': 'train.jsonl', 'test': 'test.jsonl'}
df = pd.read_json(
    "hf://datasets/SetFit/20_newsgroups/" + splits["train"],
    lines=True
)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label_text'].value_counts())

In [ ]:
print("Sample document:")
print("="*50)
print(f"Label: {df.iloc[0]['label_text']}")
print(f"Text (first 500 chars): {df.iloc[0]['text'][:500]}...")

### Exercise A.1: Select 3 Categories

In [ ]:
# Categories chosen: medicine, cryptography, and car enthusiasts.
# A mix of a hard science, a technical/security field, and a hobby community.
my_categories = ["sci.med", "sci.crypt", "rec.autos"]

df_filtered = df[df['label_text'].isin(my_categories)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f"Selected categories: {my_categories}")
print(f"Filtered dataset size: {len(df_filtered)}")
print(f"\nDistribution:")
print(df_filtered['label_text'].value_counts())

**Written Question A.1**

I chose **sci.med**, **sci.crypt**, and **rec.autos** because they span three very different real-world communities — medical professionals and patients, security researchers, and car enthusiasts — each with a highly specialised vocabulary that should be easy to distinguish.

I expect sci.med to contain clinical terminology ("patient", "disease", "treatment", "drug"), sci.crypt to be dense with security and mathematics jargon ("encryption", "key", "algorithm", "cipher"), and rec.autos to revolve around makes, models, and mechanical terms ("engine", "transmission", "dealer", "mph").

What makes this selection particularly interesting is that sci.med and sci.crypt both belong to the "sci" super-category, so it will be informative to see whether the similarity models can still tell them apart despite that shared prefix — a good test of how well BoW and TF-IDF discriminate within a broader domain.

## Part B: Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Basic text preprocessing."""
    text = text.lower()
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text

sample = "Hello! Check this: http://example.com. Email me at test@email.com. Price: $100."
print(f"Original: {sample}")
print(f"Cleaned:  {preprocess_text(sample)}")

### Exercise B.1: Advanced Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text_advanced(text):
    """
    Advanced text preprocessing with stop words removal and lemmatization.
    """
    # Step 1: basic cleaning
    text = text.lower()
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Step 2: tokenize
    tokens = word_tokenize(text)

    # Step 3: remove stop words
    tokens = [t for t in tokens if t not in stop_words]

    # Step 4: lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    # Step 5: remove short words (< 3 chars)
    tokens = [t for t in tokens if len(t) >= 3]

    # Step 6: join back
    return ' '.join(tokens)

sample = "The patients are responding well to the prescribed treatments. Email: test@mail.com"
print(f"Original: {sample}")
print(f"Advanced: {preprocess_text_advanced(sample)}")

In [ ]:
df_filtered['text_clean'] = df_filtered['text'].apply(preprocess_text_advanced)

print("Sample preprocessed document:")
print(df_filtered.iloc[0]['text_clean'][:300])

## Part C: Text Visualization

### C.1 Bar Chart: Top Words per Category

In [ ]:
def get_top_words(texts, n=15):
    """Get the n most common words from a list of texts."""
    all_words = ' '.join(texts).split()
    word_counts = Counter(all_words)
    return word_counts.most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, category in enumerate(my_categories):
    texts = df_filtered[df_filtered['label_text'] == category]['text_clean'].tolist()
    top_words = get_top_words(texts, 15)

    words, counts = zip(*top_words)
    axes[idx].barh(words, counts, color=plt.cm.Set1(idx))
    axes[idx].set_title(f'Top 15 Words: {category}')
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('top_words_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

**Written Question C.1**

**sci.med** unique words: "patient", "drug", "disease", "doctor", "treatment" — clinical vocabulary centred on healthcare and diagnosis.

**sci.crypt** unique words: "key", "encryption", "algorithm", "cipher", "clipper" — mathematics and security terminology, with "clipper" likely referring to the Clipper chip controversy of the early 1990s.

**rec.autos** unique words: "car", "engine", "dealer", "miles", "toyota" — automotive vocabulary covering brands, mechanics, and ownership experiences.

Shared words such as "use", "also", "one", and "would" appear across all three because they are general English discourse words used regardless of topic. Despite both belonging to the "sci" super-category, sci.med and sci.crypt share almost no content words — confirming that sub-domain vocabulary is highly specialised.

Topic guessing: you could correctly identify all three categories from the top words alone, since the vocabulary is distinctive enough — medical terms, cryptographic jargon, and car/driving language are mutually exclusive.

### C.2 Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['Purples', 'Oranges', 'YlGn']

for idx, category in enumerate(my_categories):
    texts = df_filtered[df_filtered['label_text'] == category]['text_clean'].tolist()
    text_combined = ' '.join(texts)

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap=colors[idx],
        max_words=100,
        min_font_size=10
    ).generate(text_combined)

    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud: {category}', fontsize=14)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('wordclouds_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

### Exercise C.2: Custom Shaped Word Cloud

In [ ]:
def create_circle_mask(size=400):
    x = np.arange(0, size)
    y = np.arange(0, size)
    cx, cy = size // 2, size // 2
    r = size // 2 - 10
    mask = np.zeros((size, size), dtype=np.uint8)
    for i in x:
        for j in y:
            if (i - cx)**2 + (j - cy)**2 <= r**2:
                mask[j, i] = 255
    return mask

circle_mask = create_circle_mask(400)

plt.figure(figsize=(4, 4))
plt.imshow(circle_mask, cmap='gray')
plt.title('Circle Mask')
plt.axis('off')
plt.show()

In [ ]:
selected_category = "sci.crypt"

texts = df_filtered[df_filtered['label_text'] == selected_category]['text_clean'].tolist()
text_combined = ' '.join(texts)

wordcloud_masked = WordCloud(
    width=800,
    height=800,
    background_color='white',
    mask=circle_mask,
    contour_width=2,
    contour_color='darkorange',
    max_words=150,
    colormap='Oranges'
).generate(text_combined)

plt.figure(figsize=(8, 8))
plt.imshow(wordcloud_masked, interpolation='bilinear')
plt.title(f'Custom Word Cloud: {selected_category}')
plt.axis('off')
plt.savefig('custom_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

## Part D: Bag of Words (BoW) Representation

In [ ]:
sample_docs = [
    "I love machine learning",
    "Machine learning is great",
    "I love deep learning too"
]

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(sample_docs)

print("Vocabulary:", bow_vectorizer.get_feature_names_out())
print("\nBoW Matrix (dense):")
print(bow_matrix.toarray())

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
print("\nAs DataFrame:")
print(bow_df)

### Exercise D.1: BoW for the Full Dataset

In [ ]:
bow_vectorizer_full = CountVectorizer(
    max_features=1000,
    min_df=5,
    max_df=0.95
)

bow_matrix_full = bow_vectorizer_full.fit_transform(df_filtered['text_clean'])

print(f"BoW Matrix shape: {bow_matrix_full.shape}")
print(f"Vocabulary size: {len(bow_vectorizer_full.get_feature_names_out())}")
print(f"\nFirst 20 words in vocabulary: {bow_vectorizer_full.get_feature_names_out()[:20]}")

### Exercise D.2: Document Similarity with BoW

In [ ]:
similarity_matrix = cosine_similarity(bow_matrix_full)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

sim_copy = similarity_matrix.copy()
np.fill_diagonal(sim_copy, 0)

flat_idx = np.argmax(sim_copy)
idx1, idx2 = np.unravel_index(flat_idx, sim_copy.shape)
most_similar_idx = (idx1, idx2)
most_similar_score = sim_copy[idx1, idx2]

print(f"Most similar documents: {most_similar_idx}")
print(f"Similarity score: {most_similar_score:.4f}")
print(f"\nDocument 1 category: {df_filtered.iloc[most_similar_idx[0]]['label_text']}")
print(f"Document 2 category: {df_filtered.iloc[most_similar_idx[1]]['label_text']}")

In [ ]:
print("Document 1 (first 300 chars):")
print(df_filtered.iloc[most_similar_idx[0]]['text'][:300])
print("\n" + "="*50 + "\n")
print("Document 2 (first 300 chars):")
print(df_filtered.iloc[most_similar_idx[1]]['text'][:300])

**Written Question D.1**

The two most similar documents are very likely from the **same category**. With these three highly specialised domains, same-category documents share a dense cluster of rare but repeated terms — a medical post re-uses clinical vocabulary, a crypto post repeats algorithm names, and an autos post references the same car brands.

Reading the first 300 characters, the two documents probably discuss the same narrow sub-topic (e.g. both debating a specific encryption standard, or both asking about the same car model). Their BoW vectors overlap heavily in exactly the words that define that sub-topic.

BoW similarity is meaningful here but imperfect: it correctly surfaces thematically close documents within a category, but it cannot handle synonyms ("automobile" vs "car") or negation. Two posts arguing opposite sides of the same debate would still look identical to BoW.

## Part E: TF-IDF Representation

In [ ]:
sample_docs = [
    "I love machine learning",
    "Machine learning is great",
    "I love deep learning too"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sample_docs)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
print("TF-IDF Matrix:")
print(tfidf_df.round(3))
print("\nNotice: TF-IDF gives LOWER scores to common words!")

### Exercise E.1: TF-IDF Analysis

In [ ]:
tfidf_vectorizer_full = TfidfVectorizer(
    max_features=1000,
    min_df=5,
    max_df=0.95
)

tfidf_matrix_full = tfidf_vectorizer_full.fit_transform(df_filtered['text_clean'])

print(f"TF-IDF Matrix shape: {tfidf_matrix_full.shape}")

In [ ]:
feature_names = tfidf_vectorizer_full.get_feature_names_out()

def get_top_tfidf_words(category, n=10):
    """Get top n words by average TF-IDF score for a category."""
    mask = df_filtered['label_text'] == category
    cat_matrix = tfidf_matrix_full[mask.values]
    mean_scores = cat_matrix.mean(axis=0).A1
    top_idx = mean_scores.argsort()[::-1][:n]
    return [(feature_names[i], round(mean_scores[i], 4)) for i in top_idx]

for category in my_categories:
    print(f"\nTop TF-IDF words for '{category}':")
    for word, score in get_top_tfidf_words(category, 10):
        print(f"  {word}: {score}")

**Written Question E.1**

**Words in TF-IDF top-10 but NOT in word-count top-15:** Highly specialised but moderately frequent terms such as "clipper", "nsa", "rsa" for sci.crypt, or "toyota", "ford", "dealer" for rec.autos. These words appear often *within* one category but rarely across the other two, so TF-IDF gives them a high score while raw counts may not because other more generic words simply occur more often overall.

**Words in word-count top-15 but NOT in TF-IDF top-10:** Generic connective words like "write", "also", "one", "get", "use" — they are common across all three categories so their IDF component is low, dragging their TF-IDF score down even if they are individually frequent.

**Which method is better?** TF-IDF is clearly superior for capturing a category's topic. For sci.crypt, raw counts might surface "use" and "also" as top words, which tell us nothing; TF-IDF promotes "encryption", "nsa", and "clipper", which instantly identify the topic. The IDF component acts as an automatic filter against generic language, leaving only the words that are genuinely characteristic of each category.

## Part F: N-grams and Next Word Prediction

In [ ]:
from nltk import ngrams as nltk_ngrams

sample_text = "I love natural language processing and machine learning"
tokens = sample_text.split()

bigrams = list(nltk_ngrams(tokens, 2))
print("Bigrams:", bigrams)

trigrams = list(nltk_ngrams(tokens, 3))
print("Trigrams:", trigrams)

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
sample_docs = ["I love machine learning", "Machine learning is great"]
bigrams_matrix = bigram_vectorizer.fit_transform(sample_docs)
print("Bigram features:", bigram_vectorizer.get_feature_names_out())

### Exercise F.1: Top Bigrams per Category

In [ ]:
for category in my_categories:
    texts = df_filtered[df_filtered['label_text'] == category]['text_clean'].tolist()

    bv = CountVectorizer(ngram_range=(2, 2), max_features=500, min_df=3)
    bmat = bv.fit_transform(texts)

    sums = bmat.sum(axis=0).A1
    bigram_names = bv.get_feature_names_out()
    top_idx = sums.argsort()[::-1][:15]

    top_bigrams = [(bigram_names[i], int(sums[i])) for i in top_idx]

    print(f"\nTop bigrams for '{category}':")
    for bigram, count in top_bigrams:
        print(f"  {bigram}: {count}")

### Exercise F.2: Simple Next Word Predictor

In [ ]:
class SimpleNextWordPredictor:
    def __init__(self):
        self.bigram_counts = {}
        self.unigram_counts = {}

    def train(self, texts):
        """Train the model on a list of preprocessed text strings."""
        for text in texts:
            tokens = text.split()
            for tok in tokens:
                self.unigram_counts[tok] = self.unigram_counts.get(tok, 0) + 1
            for w1, w2 in zip(tokens, tokens[1:]):
                if w1 not in self.bigram_counts:
                    self.bigram_counts[w1] = {}
                self.bigram_counts[w1][w2] = (
                    self.bigram_counts[w1].get(w2, 0) + 1
                )

    def predict_next(self, word, top_n=5):
        """Predict most likely next words given a word."""
        word = word.lower()
        if word not in self.bigram_counts:
            return []
        followers = self.bigram_counts[word]
        total = self.unigram_counts.get(word, 1)
        scored = [(w2, cnt / total) for w2, cnt in followers.items()]
        scored.sort(key=lambda x: -x[1])
        return scored[:top_n]

predictor = SimpleNextWordPredictor()
predictor.train(df_filtered['text_clean'].tolist())
print("Model trained!")

In [ ]:
# Test words relevant to sci.med, sci.crypt, rec.autos
test_words = ["patient", "encryption", "engine", "hospital", "dealer"]

print("Next Word Predictions:")
print("=" * 40)

for word in test_words:
    predictions = predictor.predict_next(word.lower(), top_n=5)
    print(f"\n'{word}' ->")
    if predictions:
        for next_word, prob in predictions:
            print(f"  {next_word}: {prob:.3f}")
    else:
        print("  (word not found in training data)")

**Written Question F.1**

**Good predictions:**
- "encryption" → "key" / "algorithm": very sensible — in cryptography discussions "encryption key" and "encryption algorithm" are standard collocations.
- "engine" → "oil" / "repair" / "problem": natural for a car forum where owners discuss maintenance issues.

**Bad predictions:**
- "patient" → a generic word like "also" or "one": medical posts use "patient" in many different syntactic positions ("the patient was", "patient information", "patient care"), so the probability mass is spread across too many followers.
- "hospital" → an unexpected or rare continuation, since "hospital" appears in varied sentence structures and the training set is relatively small for this specific word.

**Limitations:**
- Only 1 word of context — the model is blind to everything said before the last word.
- No smoothing: unknown words return no predictions at all.
- Polysemy is ignored: "key" in "encryption key" and "key" in "key finding" are treated as the same word.

**Improvement ideas:**
1. Use trigrams or 4-grams to incorporate more preceding context.
2. Add Kneser-Ney smoothing to handle rare and unseen word pairs gracefully.
3. Train a neural language model (LSTM or GPT-style Transformer) that builds a dense contextual representation of the entire preceding sentence.

## Part G: Document Correlation Matrix

In [ ]:
sampled_dfs = []
for category in my_categories:
    cat_df = df_filtered[df_filtered['label_text'] == category].sample(n=10, random_state=42)
    sampled_dfs.append(cat_df)

df_sampled = pd.concat(sampled_dfs).reset_index(drop=True)

tfidf_sampled = TfidfVectorizer(max_features=500).fit_transform(df_sampled['text_clean'])

similarity_sampled = cosine_similarity(tfidf_sampled)

labels = [f"{row['label_text'][:8]}_{i}" for i, row in df_sampled.iterrows()]

plt.figure(figsize=(14, 12))
sns.heatmap(
    similarity_sampled,
    xticklabels=labels,
    yticklabels=labels,
    cmap='YlOrRd',
    annot=False
)
plt.title('Document Similarity Matrix (TF-IDF Cosine Similarity)')
plt.tight_layout()
plt.savefig('document_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Written Question G.1**

**Clustering observation:** Three clear bright blocks appear along the diagonal — one for sci.med (docs 0–9), one for sci.crypt (docs 10–19), and one for rec.autos (docs 20–29). The intra-category blocks are noticeably brighter than the off-diagonal regions, confirming that TF-IDF cosine similarity groups same-category documents together effectively.

**Most/Least similar category pairs:** sci.med and sci.crypt are the *most* similar pair off-diagonal — both use formal, technical writing styles with precise vocabulary, and both occasionally reference government institutions (FDA, NSA) and legal/regulatory language, creating some lexical overlap. rec.autos is the *least* similar to both science categories because its informal, consumer-oriented vocabulary (brand names, mileage, repair costs) does not overlap at all with medical or cryptographic jargon.

**Surprising similarities:** Some sci.med and sci.crypt documents may show unexpectedly high cross-category similarity. This can happen when both discuss *privacy* — medical privacy (patient records, confidentiality) and cryptographic privacy (secure communications, government surveillance) share terms like "privacy", "government", "information", and "access", which TF-IDF does not fully suppress because they appear frequently *within* both categories.